In [44]:
from pathlib import Path
import json, csv, zipfile, shutil

In [ ]:
root = Path(r"C:/Users/<USER>/<FOLDER>/dax")
if root.exists():
    shutil.rmtree(root)
(root/'patterns'/'modeling').mkdir(parents=True)
(root/'patterns'/'security').mkdir(parents=True)
(root/'patterns'/'icons').mkdir(parents=True)
(root/'patterns'/'filter-display').mkdir(parents=True)
(root/'patterns'/'field-parameters').mkdir(parents=True)
(root/'patterns'/'navigation').mkdir(parents=True)
(root/'patterns'/'conditional-formatting').mkdir(parents=True)
(root/'patterns'/'formatting').mkdir(parents=True)
(root/'docs').mkdir(parents=True)

In [46]:
patterns = {}

In [47]:
def add(folder, code, name, description, dax, kind='value'):
    path = root/'patterns'/folder/f'{code.lower()}.dax'
    content = f'''// Pattern: {code} | {name}
// Purpose: {description}
// Required replacements:
//   [#VALUE_MEASURE#]              Existing measure evaluated by this pattern
//   <DATE_TABLE>[&DATE_FIELD&]     Continuous date column from the marked date table
// Notes:
//   Replace <#MEASURE_NAME#> with the published measure name.
//   Growth patterns return a decimal ratio and should be formatted as a percentage.
//   Validate results at year, quarter, month, and total levels before release.

{dax.strip()}
'''
    print(path)
    path.write_text(content, encoding='utf-8')
    patterns[code] = {'code':code,'name':name,'description':description,'kind':kind,'path':str(path.relative_to(root)).replace('\\','/')}

In [ ]:
# ---------------------------------------------------------------------------
# Formatting
# ---------------------------------------------------------------------------

add(
    "formatting",
    "CUSTOM_SPACING",
    "Custom Spacing",
    "Aligns two formatted measure values by inserting invisible spacing characters.",
    """
<#MEASURE_NAME#> =
VAR _spacing =
    "‏‏‎ ‎"
VAR _length =
    LEN(
        FORMAT(
            [#DETAIL_MEASURE#],
            "#%"
        )
    )
VAR _needed =
    IF(
        _length >= 4,
        -2,
        IF(
            _length >= 3,
            0,
            IF(
                _length = 1,
                4,
                2
            )
        )
    )
    + IF(
        _length = 0,
        10,
        5
    )
VAR _result =
    FORMAT(
        [#VALUE_MEASURE#],
        "#,##.0"
    )
    & REPT(
        _spacing,
        MAX(
            0,
            _needed
        )
    )
    & FORMAT(
        [#DETAIL_MEASURE#],
        "#%"
    )
RETURN
    _result
"""
)

# ---------------------------------------------------------------------------
# Conditional Formatting
# ---------------------------------------------------------------------------

add(
    "conditional-formatting",
    "DIVERGENT_COLOR",
    "Divergent Color",
    "Returns a color based on the selected-range position of the current measure value.",
    """
<#MEASURE_NAME#> =
VAR _current =
    [#VALUE_MEASURE#]

VAR _minimum =
    IF(
        NOT ISFILTERED(
            <CATEGORY_TABLE>[&CATEGORY_FIELD&]
        ),
        MINX(
            CALCULATETABLE(
                VALUES(
                    <DATE_TABLE>[&PERIOD_FIELD&]
                ),
                ALLSELECTED(
                    <DATE_TABLE>[&PERIOD_FIELD&]
                )
            ),
            [#VALUE_MEASURE#]
        ),
        MINX(
            CALCULATETABLE(
                VALUES(
                    <CATEGORY_TABLE>[&CATEGORY_FIELD&]
                ),
                ALLSELECTED(
                    <CATEGORY_TABLE>[&CATEGORY_FIELD&]
                )
            ),
            MINX(
                CALCULATETABLE(
                    VALUES(
                        <DATE_TABLE>[&PERIOD_FIELD&]
                    ),
                    ALLSELECTED(
                        <DATE_TABLE>[&PERIOD_FIELD&]
                    )
                ),
                [#VALUE_MEASURE#]
            )
        )
    )

VAR _maximum =
    IF(
        NOT ISFILTERED(
            <CATEGORY_TABLE>[&CATEGORY_FIELD&]
        ),
        MAXX(
            CALCULATETABLE(
                VALUES(
                    <DATE_TABLE>[&PERIOD_FIELD&]
                ),
                ALLSELECTED(
                    <DATE_TABLE>[&PERIOD_FIELD&]
                )
            ),
            [#VALUE_MEASURE#]
        ),
        MAXX(
            CALCULATETABLE(
                VALUES(
                    <CATEGORY_TABLE>[&CATEGORY_FIELD&]
                ),
                ALLSELECTED(
                    <CATEGORY_TABLE>[&CATEGORY_FIELD&]
                )
            ),
            MAXX(
                CALCULATETABLE(
                    VALUES(
                        <DATE_TABLE>[&PERIOD_FIELD&]
                    ),
                    ALLSELECTED(
                        <DATE_TABLE>[&PERIOD_FIELD&]
                    )
                ),
                [#VALUE_MEASURE#]
            )
        )
    )

VAR _normalized =
    DIVIDE(
        _current - _minimum,
        _maximum - _minimum,
        0.5
    )

VAR _result =
    SWITCH(
        TRUE(),
        ISBLANK(_current), BLANK(),
        _normalized <= 0.10, "#BCE4D8",
        _normalized <= 0.20, "#7EC1CA",
        _normalized <= 0.30, "#3D98B3",
        _normalized <= 0.40, "#3689AA",
        _normalized <= 0.50, "#95ACC2",
        _normalized <= 0.60, "#809BB5",
        _normalized <= 0.70, "#6B8AA9",
        _normalized <= 0.80, "#567A9D",
        _normalized <= 0.90, "#416991",
        "#2C5985"
    )

RETURN
    _result
"""
)

# ---------------------------------------------------------------------------
# Navigation
# ---------------------------------------------------------------------------

add(
    "navigation",
    "DRILL_THROUGH_MONTHLY_TEXT",
    "Drill Through Monthly Text",
    "Returns monthly drill-through text when a parameter is selected.",
    """
<#MEASURE_NAME#> =
VAR _selected =
    SELECTEDVALUE(
        <PARAMETER_TABLE>[&PARAMETER_FIELD&]
    )

VAR _single =
    HASONEVALUE(
        <DATE_TABLE>[&DATE_FIELD&]
    )

VAR _result =
    IF(
        NOT ISBLANK(_selected)
            && NOT _single,
        "Monthly "
            & _selected
            & " Data"
    )

RETURN
    _result
"""
)

add(
    "navigation",
    "DRILL_THROUGH_DAILY_TEXT",
    "Drill Through Daily Text",
    "Returns daily drill-through text when a parameter is selected.",
    """
<#MEASURE_NAME#> =
VAR _selected =
    SELECTEDVALUE(
        <PARAMETER_TABLE>[&PARAMETER_FIELD&]
    )

VAR _result =
    IF(
        NOT ISBLANK(_selected),
        "Drill to Daily "
            & _selected
            & " Data"
    )

RETURN
    _result
"""
)

# ---------------------------------------------------------------------------
# Field Parameters
# ---------------------------------------------------------------------------

add(
    "field-parameters",
    "MEASURE_SELECTION",
    "Measure Selection",
    "Field parameter for switching between published measures.",
    """
<#PARAMETER_NAME#> =
{
    (
        "<LABEL_1>",
        NAMEOF(
            <MEASURE_TABLE>[#MEASURE_1#]
        ),
        0
    ),
    (
        "<LABEL_2>",
        NAMEOF(
            <MEASURE_TABLE>[#MEASURE_2#]
        ),
        1
    )
}
""",
    kind="field parameter"
)

# ---------------------------------------------------------------------------
# Filter Display
# ---------------------------------------------------------------------------

add(
    "filter-display",
    "MONTH_FILTER_CARD",
    "Month Filter Card",
    "Displays selected months in a card visual.",
    """
<#MEASURE_NAME#> =
VAR _limit = 1

VAR _filter =
    FILTERS(
        <DATE_TABLE>[&MONTH_FILTER_FIELD&]
    )

VAR _count =
    COUNTROWS(
        _filter
    )

VAR _top =
    TOPN(
        _limit,
        _filter,
        <DATE_TABLE>[&MONTH_FILTER_FIELD&],
        ASC
    )

VAR _text =
    CONCATENATEX(
        _top,
        <DATE_TABLE>[&MONTH_FILTER_FIELD&],
        ", "
    )

VAR _result =
    IF(
        ISFILTERED(
            <DATE_TABLE>[&MONTH_FILTER_FIELD&]
        ),
        "Selected Month = "
            & _text
            & IF(
                _count > _limit,
                ", ... ["
                    & _count
                    & " items selected]"
            )
            & " "
            & UNICHAR(13)
            & UNICHAR(10)
    )

RETURN
    _result
"""
)

# ---------------------------------------------------------------------------
# Security
# ---------------------------------------------------------------------------

add(
    "security",
    "SENSITIVE_DATA",
    "Sensitive Data",
    "Returns data only when the required permission exists.",
    """
<#MEASURE_NAME#> =
VAR _allowed =
    HASPERMISSION(
        "<PERMISSION_NAME>"
    )

VAR _result =
    IF(
        _allowed,
        [#VALUE_MEASURE#],
        "REDACTED"
    )

RETURN
    _result
"""
)

# ---------------------------------------------------------------------------
# ICONS
# ---------------------------------------------------------------------------

add(
    "icons",
    "PERIOD_CHANGE_ICON",
    "Period Change Icon",
    "Compares current MTD to prior month and returns an icon.",
    """
<#MEASURE_NAME#> =
VAR _current =
    [#VALUE_MEASURE#]

VAR _previous =
    CALCULATE(
        [#VALUE_MEASURE#],
        DATEADD(
            DATESMTD(
                <DATE_TABLE>[&DATE_FIELD&]
            ),
            -1,
            MONTH
        )
    )

VAR _change =
    _current - _previous

VAR _result =
    SWITCH(
        TRUE(),
        ISBLANK(_change),
            BLANK(),
        _change > 0,
            "✔ "
                & FORMAT(
                    _change,
                    "#.##%"
                ),
        _change = 0,
            "",
        "🔻 "
            & FORMAT(
                _change,
                "#.##%"
            )
    )

RETURN
    _result
"""
)

add(
  "icons",
  "CARD_COLUMNS_NO_TARGET",
  "Card with Columns No Target",
  "Returns an SVG card with a slicer-aware callout and a trailing-month column chart.",
  r'''<#MEASURE_NAME#> =
/* =========================
   SETUP
========================= */
VAR _space = 3
VAR _maximumWidth = 6
VAR _rounding = 4
VAR _width = 400
VAR _height = 150
/* =========================
   COLORS
========================= */
VAR _barColor = "#004081"
VAR _textColor = "#73AF2F"
VAR _labelColor = "#333333"
/* =========================
   SVG WRAPPER
========================= */
VAR _declaration = "data:image/svg+xml;utf8,"
VAR _header =
  "<svg xmlns='http://www.w3.org/2000/svg' width='"
    & _width & "' height='" & _height & "'>"
VAR _end = "</svg>"
VAR _style =
  "<style>text{font-family:Segoe UI;dominant-baseline:middle}.bar{fill:"
    & _barColor & ";}.bolder{font-weight:600;}</style>"
/* =========================
   CALLOUT
========================= */
VAR _selectedValue = [#VALUE_MEASURE#]
VAR _selectedPeriod = SELECTEDVALUE(<DATE_TABLE>[&PERIOD_FIELD&])
VAR _callout =
  "<text x='20%' y='15%' fill='" & _textColor & "' font-size='12'>"
    & "<tspan class='bolder'>" & FORMAT(_selectedValue, "#,0") & "</tspan>"
    & "<tspan font-size='12'> <CALLOUT_SUFFIX> " & _selectedPeriod & "</tspan></text>"
/* =========================
   BAR DATA
========================= */
VAR _anchor =
  CALCULATE(
    MAX(<DATE_TABLE>[&PERIOD_SORT_FIELD&]),
    REMOVEFILTERS(<DATE_TABLE>)
  )
VAR _periods =
  FILTER(
    SUMMARIZE(
      ALL(<DATE_TABLE>),
      <DATE_TABLE>[&PERIOD_FIELD&],
      <DATE_TABLE>[&PERIOD_SORT_FIELD&]
    ),
    <DATE_TABLE>[&PERIOD_SORT_FIELD&] <= _anchor
      && <DATE_TABLE>[&PERIOD_SORT_FIELD&] > _anchor - <PERIOD_COUNT>
  )
VAR _data =
  ADDCOLUMNS(
    _periods,
    "@Value", [#VALUE_MEASURE#],
    "@Row",
      ROWNUMBER(
        _periods,
        ORDERBY(<DATE_TABLE>[&PERIOD_SORT_FIELD&], ASC)
      )
  )
VAR _count = COUNTROWS(_data)
VAR _maximum = MAXX(_data, [@Value])
VAR _columnWidth =
  MIN(
    _maximumWidth,
    DIVIDE(100 - (_count + 1) * _space, _count)
  )
/* =========================
   BARS
========================= */
VAR _bars =
  CONCATENATEX(
    _data,
    VAR _barHeight =
      ROUND(DIVIDE([@Value], _maximum, 0) * 45, 2)
    VAR _x = ([@Row] - 1) * _columnWidth + ([@Row] * _space)
    VAR _y = 80 - _barHeight
    VAR _centerX = _x + (_columnWidth / 2)
    VAR _centerY = _y + (_barHeight / 2)
    RETURN
      "<rect class='bar' x='" & _x & "%' y='" & _y
        & "%' width='" & _columnWidth & "%' height='" & _barHeight
        & "%' rx='" & _rounding & "'/>"
        & IF(
            _barHeight >= 8,
            "<text x='" & _centerX & "%' y='" & _centerY
              & "%' text-anchor='middle' font-size='10' fill='white'>"
              & FORMAT([@Value], "#,0") & "</text>",
            ""
          )
  )
/* =========================
   PERIOD LABELS
========================= */
VAR _labels =
  CONCATENATEX(
    _data,
    VAR _centerX =
      ([@Row] - 1) * _columnWidth
        + ([@Row] * _space)
        + (_columnWidth / 2)
    RETURN
      "<text x='" & _centerX
        & "%' y='86%' text-anchor='middle' font-size='10' fill='"
        & _labelColor & "'>"
        & <DATE_TABLE>[&PERIOD_LABEL_FIELD&]
        & "</text>"
  )
RETURN
  _declaration & _header & _style & _callout & _bars & _labels & _end'''
)


add(
  "icons",
  "CARD_COLUMNS_TARGET",
  "Card with Columns and Target",
  "Returns an SVG KPI card with columns, a target line, period labels, and summary text.",
  r'''<#MEASURE_NAME#> =
VAR _space = 5
VAR _maximumColumnWidth = 400
VAR _rounding = 5
VAR _width = 413.33
VAR _height = 290
VAR _textHeight = _height * 0.30
VAR _data =
  ADDCOLUMNS(
    SUMMARIZE(
      <DATE_TABLE>,
      <DATE_TABLE>[&PERIOD_FIELD&],
      <DATE_TABLE>[&PERIOD_SORT_FIELD&]
    ),
    "@Value", [#VALUE_MEASURE#],
    "@Target", [#TARGET_MEASURE#],
    "@Variance", [#VARIANCE_MEASURE#]
  )
VAR _prepared =
  ADDCOLUMNS(
    _data,
    "@Row",
      ROWNUMBER(
        _data,
        ORDERBY(<DATE_TABLE>[&PERIOD_SORT_FIELD&], ASC)
      )
  )
VAR _maximumValue = MAXX(_prepared, [@Value])
VAR _maximumTarget = MAXX(_prepared, [@Target])
VAR _scaleMaximum = MAX(_maximumValue, _maximumTarget)
VAR _count = COUNTROWS(_prepared)
VAR _columnWidth =
  MIN(
    _maximumColumnWidth,
    ROUND(DIVIDE(_width - ((_count + 1) * _space), _count), 1)
  )
VAR _columns =
  CONCATENATEX(
    _prepared,
    VAR _color = IF([@Variance] >= 0, "#FF1C0E", "#26FF07")
    VAR _barHeight = ROUND(DIVIDE([@Value], _scaleMaximum, 0) * (_height * 0.60), 1)
    VAR _x = (([@Row] - 1) * _columnWidth) + (_space * [@Row])
    VAR _y = _textHeight + (_height * 0.60) - _barHeight
    RETURN
      "<rect x='" & _x & "' y='" & _y & "' width='" & _columnWidth
        & "' height='" & _barHeight & "' fill='" & _color
        & "' rx='" & _rounding & "'/>"
  )
VAR _targetPoints =
  CONCATENATEX(
    _prepared,
    VAR _x = (([@Row] - 1) * _columnWidth) + (_space * [@Row]) + (_columnWidth / 2)
    VAR _y = _textHeight + (_height * 0.60) - DIVIDE([@Target], _scaleMaximum, 0) * (_height * 0.60)
    RETURN FORMAT(_x, "0") & "," & FORMAT(_y, "0"),
    " ",
    <DATE_TABLE>[&PERIOD_SORT_FIELD&], ASC
  )
VAR _line =
  "<polyline points='" & _targetPoints
    & "' stroke='#333333' stroke-width='4' fill='none' stroke-linejoin='round'"
    & " stroke-linecap='round' stroke-dasharray='8,8'/>"
VAR _labels =
  CONCATENATEX(
    _prepared,
    VAR _x = (([@Row] - 1) * _columnWidth) + (_space * [@Row]) + (_columnWidth / 2)
    RETURN
      "<text x='" & _x & "' y='95%' font-size='10' text-anchor='middle' fill='#808080'>"
        & <DATE_TABLE>[&PERIOD_FIELD&] & "</text>"
  )
VAR _summary =
  "<text x='4%' y='8%' font-family='Segoe UI' font-size='18' font-weight='600' fill='#333333'>"
    & "<TITLE_TEXT>" & "</text>"
    & "<text x='99%' y='8%' text-anchor='end' font-family='Segoe UI' font-size='12' fill='#808080'>"
    & [#SUMMARY_MEASURE#] & "</text>"
RETURN
  "data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' width='"
    & _width & "' height='" & _height & "'>"
    & _summary & _columns & _line & _labels & "</svg>"'''
)


add(
  "icons",
  "CARD_TEXT_ONLY",
  "Card with Text Only",
  "Returns an SVG KPI card containing primary, secondary, and target-summary text.",
  r'''<#MEASURE_NAME#> =
VAR _width = 413.33
VAR _height = 80
VAR _data =
  ADDCOLUMNS(
    SUMMARIZE(
      <DATE_TABLE>,
      <DATE_TABLE>[&PERIOD_FIELD&],
      <DATE_TABLE>[&PERIOD_SORT_FIELD&]
    ),
    "@Value", [#VALUE_MEASURE#],
    "@Target", [#TARGET_MEASURE#]
  )
VAR _count = COUNTROWS(_data)
VAR _met = COUNTROWS(FILTER(_data, [@Target] >= [@Value]))
VAR _style =
  "<style>text{font-family:Segoe UI}.bold{font-weight:600}</style>"
VAR _content =
  "<text x='4%' y='24%' font-size='18' fill='#333333' class='bold'><TITLE_TEXT></text>"
    & "<text x='99%' y='20%' text-anchor='end' font-size='12' fill='#808080'>"
    & [#PRIMARY_SUMMARY_MEASURE#] & "</text>"
    & "<text x='4%' y='74%' font-size='18' fill='#333333' class='bold'>"
    & [#SECONDARY_SUMMARY_MEASURE#] & "</text>"
    & "<text x='99%' y='50%' text-anchor='end' font-size='12' fill='#808080'>"
    & [#PERIOD_SUMMARY_MEASURE#] & "</text>"
    & "<text x='99%' y='80%' text-anchor='end' font-size='10' fill='#B3B3B3'>"
    & _met & " of " & _count & " <TARGET_SUFFIX></text>"
RETURN
  "data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' width='"
    & _width & "' height='" & _height & "'>"
    & _style & _content & "</svg>"'''
)


add(
  "icons",
  "NOTIFICATION_ICON",
  "Notification Icon",
  "Returns an SVG notification bell with a badge showing the number of actionable groups.",
  r'''<#MEASURE_NAME#> =
VAR _groups =
  FILTER(
    VALUES(<GROUP_TABLE>[&GROUP_FIELD&]),
    <GROUP_TABLE>[&GROUP_FIELD&]
      IN VALUES(<STATUS_TABLE>[&STATUS_GROUP_FIELD&])
  )
VAR _status =
  ADDCOLUMNS(
    _groups,
    "@EndDate",
      CALCULATE(
        MAX(<STATUS_TABLE>[&END_DATE_FIELD&]),
        FILTER(
          ALL(<STATUS_TABLE>),
          <STATUS_TABLE>[&STATUS_GROUP_FIELD&]
            = <GROUP_TABLE>[&GROUP_FIELD&]
        )
      ),
    "@ComparisonDate",
      CALCULATE(
        MAX(<STATUS_TABLE>[&COMPARISON_DATE_FIELD&]),
        FILTER(
          ALL(<STATUS_TABLE>),
          <STATUS_TABLE>[&STATUS_GROUP_FIELD&]
            = <GROUP_TABLE>[&GROUP_FIELD&]
        )
      )
  )
VAR _count = COUNTROWS(FILTER(_status, [@ComparisonDate] <> [@EndDate]))
VAR _bell =
  "<path fill-rule='evenodd' clip-rule='evenodd' d='M13.8094 49.875C13.8094 22.33 36.1394 0 63.6844 0C91.2294 0 113.559 22.33 113.559 49.875V76.6935L126.541 102.657C128.93 107.435 125.456 114 119.53 114H91.2867C88.079 126.37 76.859 135.36 63.6844 135.36C50.5098 135.36 39.2898 126.37 36.0822 114H7.83866C1.912 114 -1.562 107.435 0.82766 102.657L13.8094 76.6935V49.875ZM18.2198 99.75H109.156L100.82 83.0632C99.829 81.086 99.3094 78.9052 99.3094 76.6935V49.875C99.3094 30.198 83.361 14.25 63.6844 14.25C44.0078 14.25 28.0594 30.198 28.0594 49.875V76.6935C28.0594 78.9046 27.5446 81.0854 26.556 83.0632L18.2198 99.75Z' fill='#808080'/>"
VAR _badge =
  IF(
    _count > 0,
    "<circle cx='100' cy='40' r='28' fill='#C00000'/>"
      & "<text x='100' y='50' font-size='32' text-anchor='middle' fill='white' font-weight='bold'>"
      & _count & "</text>",
    ""
  )
RETURN
  IF(
    _count > 0,
    "data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' width='100' height='100' viewBox='0 0 128 156'>"
      & _bell & _badge & "</svg>"
  )'''
)


add(
  "icons",
  "STAR_RATING_SVG",
  "Star Rating SVG",
  "Returns an SVG rating made from full, half, and empty stars.",
  r'''<#MEASURE_NAME#> =
VAR _metric = SELECTEDVALUE(<THRESHOLD_TABLE>[&METRIC_FIELD&])
VAR _score = [#VALUE_MEASURE#]
VAR _maximum =
  CALCULATE(
    MAX(<WEIGHT_TABLE>[&WEIGHT_FIELD&]),
    <WEIGHT_TABLE>[&METRIC_FIELD&] = _metric
  )
VAR _percentage = COALESCE(DIVIDE(_score, _maximum), 0)
VAR _total = <STAR_COUNT>
VAR _exact = MIN(_total, MAX(0, _percentage * _total))
VAR _fullCount = INT(_exact)
VAR _halfCount = IF(_exact - _fullCount >= 0.5, 1, 0)
VAR _emptyCount = _total - _fullCount - _halfCount
VAR _full =
  "<path fill='#9C6500' d='M62.615 5.36c3.527-7.147 13.723-7.147 17.25 0l16.8 34.043 37.57 5.458c7.888 1.14 11.037 10.844 5.33 16.409l-27.189 26.497 6.419 37.407c1.354 7.866-6.897 13.858-13.958 10.146l-33.601-17.67-33.594 17.67c-7.054 3.705-15.305-2.28-13.965-10.139l6.42-37.413L2.913 61.263C-2.793 55.705.356 46.008 8.244 44.86l37.57-5.458 16.8-34.043Z'/>"
VAR _half =
  "<defs><linearGradient id='half'><stop offset='50%' stop-color='#9C6500'/><stop offset='50%' stop-color='white'/></linearGradient></defs>"
    & SUBSTITUTE(_full, "fill='#9C6500'", "fill='url(#half)' stroke='#9C6500' stroke-width='4'")
VAR _empty = SUBSTITUTE(_full, "fill='#9C6500'", "fill='none' stroke='#9C6500' stroke-width='4'")
VAR _cellWidth = 150
VAR _cellHeight = 150
VAR _fullStars =
  CONCATENATEX(
    GENERATESERIES(0, _fullCount - 1, 1),
    "<svg x='" & (_cellWidth * [Value])
      & "' width='" & _cellWidth & "' height='" & _cellHeight
      & "' viewBox='0 0 150 150'>" & _full & "</svg>",
    ""
  )
VAR _halfStar =
  IF(
    _halfCount = 1,
    "<svg x='" & (_cellWidth * _fullCount)
      & "' width='" & _cellWidth & "' height='" & _cellHeight
      & "' viewBox='0 0 150 150'>" & _half & "</svg>",
    ""
  )
VAR _emptyStars =
  CONCATENATEX(
    GENERATESERIES(0, _emptyCount - 1, 1),
    "<svg x='" & (_cellWidth * (_fullCount + _halfCount + [Value]))
      & "' width='" & _cellWidth & "' height='" & _cellHeight
      & "' viewBox='0 0 150 150'>" & _empty & "</svg>",
    ""
  )
RETURN
  "data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 "
    & (_cellWidth * _total) & " " & _cellHeight & "' width='300' height='30'>"
    & _fullStars & _halfStar & _emptyStars & "</svg>"'''
)


add(
  "icons",
  "CALENDAR_DETAILS_SVG",
  "Calendar with Details SVG",
  "Returns an SVG calendar grid with completed, qualifying, and remaining-day details.",
  r'''<#MEASURE_NAME#> =
VAR _days = MAX(<DATE_TABLE>[&DAYS_IN_PERIOD_FIELD&])
VAR _completed = MAX(<DATE_TABLE>[&DAY_NUMBER_FIELD&])
VAR _firstDate = MAX(<DATE_TABLE>[&PERIOD_FIRST_DATE_FIELD&])
VAR _firstWeekday =
  CALCULATE(
    MAX(<DATE_TABLE>[&WEEKDAY_NUMBER_FIELD&]),
    <DATE_TABLE>[&DATE_FIELD&] = _firstDate
  )
VAR _qualifyingCompleted = SUM(<DATE_TABLE>[&QUALIFYING_DAY_FIELD&])
VAR _qualifyingTotal = MAX(<DATE_TABLE>[&QUALIFYING_DAYS_IN_PERIOD_FIELD&])
VAR _percentage = DIVIDE(_qualifyingCompleted, _qualifyingTotal, 0)
VAR _space = 5
VAR _width = 550
VAR _height = 200
VAR _padding = 15
VAR _columns = 7
VAR _rows = ROUNDUP(DIVIDE(_days + _firstWeekday - 1, _columns), 0)
VAR _cube =
  MIN(
    DIVIDE(_height - (_padding * 2), _rows + 1),
    DIVIDE(_width - 130, _columns + 1)
  )
VAR _gradient =
  "<defs><linearGradient id='pending' x1='0%' y1='0%' x2='0%' y2='100%'>"
    & "<stop offset='0%' stop-color='#386AA5' stop-opacity='.8'/>"
    & "<stop offset='85%' stop-color='#CCCCCC'/></linearGradient></defs>"
VAR _series = SELECTCOLUMNS(GENERATESERIES(1, _days, 1), "Day", [Value])
VAR _cubes =
  CONCATENATEX(
    _series,
    VAR _day = [Day]
    VAR _index = _day + _firstWeekday - 2
    VAR _row = INT(DIVIDE(_index, _columns))
    VAR _column = MOD(_index, _columns)
    VAR _x = (_column * (_cube + _space)) + _padding
    VAR _y = (_row * (_cube + _space)) + _padding
    VAR _date = _firstDate + (_day - 1)
    VAR _qualifies =
      LOOKUPVALUE(
        <DATE_TABLE>[&QUALIFYING_DAY_FIELD&],
        <DATE_TABLE>[&DATE_FIELD&], _date
      )
    VAR _fill =
      SWITCH(
        TRUE(),
        _day > _completed, "#CCCCCC",
        _qualifies = 1, "#386AA5",
        "url(#pending)"
      )
    RETURN
      "<rect x='" & _x & "' y='" & _y & "' width='" & _cube
        & "' height='" & _cube & "' fill='" & _fill & "'/>"
  )
VAR _remaining = _qualifyingTotal - _qualifyingCompleted
VAR _textX = (_columns * (_cube + _space)) + _padding + 15
VAR _details =
  "<text x='" & _textX & "' y='30' fill='#386AA5' font-family='Segoe UI' font-size='18'>"
    & FORMAT(_percentage, "0.0%") & " Completed</text>"
    & "<text x='" & _textX & "' y='50' fill='#898E8F' font-family='Segoe UI' font-size='16'>Completed: "
    & FORMAT(_qualifyingCompleted, "#,0.0") & "</text>"
    & "<text x='" & _textX & "' y='70' fill='#898E8F' font-family='Segoe UI' font-size='16'>Total: "
    & FORMAT(_qualifyingTotal, "#,0.0") & "</text>"
    & "<text x='" & _textX & "' y='90' fill='#898E8F' font-family='Segoe UI' font-size='16'>Remaining: "
    & FORMAT(_remaining, "#,0.0") & "</text>"
RETURN
  "data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' height='"
    & _height & "' width='" & _width & "'>"
    & _gradient & _cubes & _details & "</svg>"'''
)


add(
  "icons",
  "DONUT_TARGET_PROGRESS_SVG",
  "Donut Target Progress SVG",
  "Returns an SVG donut showing target, current value, and the amount over or under target.",
  r'''<#MEASURE_NAME#> =
VAR _target = [#TARGET_MEASURE#]
VAR _value = COALESCE([#VALUE_MEASURE#], 0)
VAR _width = 365
VAR _height = 280.27
VAR _canvas = MIN(_height, _width)
VAR _padding = 10
VAR _label = IF(_value >= _target, "Over:", "Missing:")
VAR _positive = "#006100"
VAR _neutral = "#9C6500"
VAR _negative = "#9C0006"
VAR _half = _canvas / 2
VAR _radius = _half - _padding * 1.5
VAR _circumference = 2 * PI() * _radius
VAR _percentage = DIVIDE(_value, _target, 0)
VAR _offset = _circumference - _circumference * MIN(_percentage, 1)
VAR _color =
  SWITCH(
    TRUE(),
    _percentage >= 0.90, _positive,
    _percentage >= 0.80, _neutral,
    _negative
  )
VAR _style =
  "<style>text{font-family:Segoe UI}.circle{fill:transparent;stroke-width:"
    & _padding & ";transform:rotate(-90deg);transform-origin:50% 50%}"
    & ".background{stroke:#E0E0E0}.filling{stroke:" & _color
    & ";stroke-linecap:round;stroke-dasharray:" & _circumference
    & ";stroke-dashoffset:" & _offset & "}</style>"
VAR _content =
  "<circle r='" & _radius & "' cx='50%' cy='50%' class='circle background'/>"
    & "<circle r='" & _radius & "' cx='50%' cy='50%' class='circle filling'/>"
    & "<text x='50%' y='38%' text-anchor='middle' font-size='42' font-weight='bold' fill='"
    & _color & "'>" & FORMAT(_percentage, "0.0%") & "</text>"
    & "<text x='48%' y='52%' text-anchor='end' font-size='18'>Target:</text>"
    & "<text x='52%' y='52%' font-size='18'>" & FORMAT(_target, "#,#") & "</text>"
    & "<text x='48%' y='62%' text-anchor='end' font-size='18'>Current:</text>"
    & "<text x='52%' y='62%' font-size='18'>" & FORMAT(_value, "#,#") & "</text>"
    & "<text x='48%' y='72%' text-anchor='end' font-size='18' font-weight='bold'>"
    & _label & "</text>"
    & "<text x='52%' y='72%' font-size='18' font-weight='bold'>"
    & FORMAT(ABS(_target - _value), "#,#") & "</text>"
RETURN
  IF(
    NOT ISBLANK(_target),
    "data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' height='"
      & _height & "' width='" & _width & "'>"
      & _style & _content & "</svg>"
  )'''
)

# ---------------------------------------------------------------------------
# Modeling
# ---------------------------------------------------------------------------

add(
    "modeling",
    "ADD_SYNTHETIC_ROW",
    "Add Synthetic Row",
    "Adds a synthetic row to an existing table.",
    """
<#TABLE_NAME#> =
DISTINCT(
    UNION(
        SELECTCOLUMNS(
            <SOURCE_TABLE>,
            "<CATEGORY_COLUMN>",
                <SOURCE_TABLE>[&CATEGORY_FIELD&],
            "<LINE_COLUMN>",
                <SOURCE_TABLE>[&LINE_FIELD&],
            "<NAME_COLUMN>",
                <SOURCE_TABLE>[&NAME_FIELD&],
            "<SORT_COLUMN>",
                <SOURCE_TABLE>[&SORT_FIELD&],
            "<FLAG_COLUMN>",
                FALSE()
        ),
        DATATABLE(
            "<CATEGORY_COLUMN>", STRING,
            "<LINE_COLUMN>", STRING,
            "<NAME_COLUMN>", STRING,
            "<SORT_COLUMN>", INTEGER,
            "<FLAG_COLUMN>", BOOLEAN,
            {
                {
                    "<SYNTHETIC_LABEL>",
                    BLANK(),
                    BLANK(),
                    <SORT_VALUE>,
                    TRUE()
                }
            }
        )
    )
)
""",
    kind="calculated table"
)


In [49]:
readme = '''# DAX Pattern Templates

A Git-friendly library of reusable, commented DAX patterns for Power BI and Fabric semantic models. Each pattern is stored in its own `.dax` file so changes are easy to review, test, and merge.

## Included pattern categories

- Period to Date
- Moving Periods
- Previous Periods
- Growth
- Dynamic Calculations
- Modeling
- Security
- Icons and Indicators
- Filter Display
- Field Parameters
- Navigation
- Conditional Formatting
- Formatting

## Placeholder contract

- `[#VALUE_MEASURE#]`: Existing measure evaluated by the pattern.
- `[#DETAIL_MEASURE#]`: Supporting detail measure used by formatting patterns.
- `<#MEASURE_NAME#>`: Published measure name.
- `<#TABLE_NAME#>`: Published calculated table name.
- `<#PARAMETER_NAME#>`: Published field parameter name.
- `<DATE_TABLE>`: Marked date table.
- `[&DATE_FIELD&]`: Continuous date column.
- `<CATEGORY_TABLE>`: Table containing the comparison category.
- `[&CATEGORY_FIELD&]`: Category field used by the visual.
- `<PARAMETER_TABLE>`: Field parameter table.
- `[&PARAMETER_FIELD&]`: Display field from the parameter table.
- `<MEASURE_TABLE>`: Table containing referenced measures.

Example replacement:

```text
<#MEASURE_NAME#>            -> Revenue YTD
[#VALUE_MEASURE#]          -> [Revenue]
<DATE_TABLE>[&DATE_FIELD&] -> 'dimDates'[DateValue]
<CATEGORY_TABLE>           -> 'dimProvider'
[&CATEGORY_FIELD&]         -> [Provider Name]
```

## Repository Workflow

1. Create a branch named `feature/<pattern-or-change>`.
2. Edit one pattern per file whenever practical.
3. Update documentation when behavior changes.
4. Validate the DAX against the checklist in `docs/testing.md`.
5. Open a pull request and include the tested model, date range, and expected result.

## Design Decisions

- Reusable placeholders are preferred over hard-coded model references.
- Time-intelligence templates assume a properly configured marked date table.
- Percentage calculations should use `DIVIDE` for safe zero-denominator handling.
- Formatting and icon patterns return text and should not be used in numeric calculations.
- Conditional formatting patterns return values intended for field-value conditional formatting.
- Security patterns are intended as display-layer controls and do not replace Row-Level Security.
- Field-parameter templates are stored separately from standard measure templates.
- Calculated-table patterns clearly identify themselves in metadata and comments.

## Folder Structure

```text
patterns/
├── period-to-date/
├── moving/
├── previous-period/
├── growth/
├── dynamic/
├── modeling/
├── security/
├── icons/
├── filter-display/
├── field-parameters/
├── navigation/
├── conditional-formatting/
└── formatting/

docs/
```

## Validation Checklist

Before publishing a pattern:

- Replace all placeholders.
- Confirm referenced measures exist.
- Verify date table configuration.
- Test row-level values.
- Test subtotals.
- Test grand totals.
- Test slicer interactions.
- Validate blank handling.
- Validate zero-denominator handling.
- Confirm expected formatting.
- Confirm the pattern returns the documented data type.

## Status

This repository is provided as an internal template library. 
'''

(root/'README.md').write_text(readme, encoding='utf-8')
(root/'CONTRIBUTING.md').write_text('''# Contributing

## Pull request requirements

- Keep placeholders unchanged unless the placeholder contract is intentionally revised.
- Preserve comments that explain purpose, replacements, assumptions, and output type.
- Use underscore-prefixed, singular DAX variable names with proper casing.
- Use `DIVIDE` for ratios.
- Add or update validation notes when calculation behavior changes.
- Do not mix fiscal and calendar behavior in the same template.

## Review checklist

- [ ] DAX parses successfully.
- [ ] Current-period result matches a manually verified result.
- [ ] Prior-period result is aligned to the intended comparison window.
- [ ] Totals and hierarchy levels behave as documented.
- [ ] Blank and zero comparison values do not produce errors.
- [ ] Date filters outside the target period do not leak into complete-period results.
''', encoding='utf-8')
(root/'CHANGELOG.md').write_text('''# Changelog

## 0.1.0 - Initial library

- Added 25 reusable time-intelligence patterns.
- Added 15 reusable dax patterns. 
- Added placeholder contract, contribution guidance, and testing checklist.
''', encoding='utf-8')
(root/'VERSION').write_text('0.1.0\n', encoding='utf-8')
(root/'.gitignore').write_text('''*.tmp
*.bak
.DS_Store
Thumbs.db
.vscode/
''', encoding='utf-8')
(root/'docs'/'testing.md').write_text('''# Pattern Testing

Test each pattern with a small, independently verifiable base measure.

## Required checks

1. Confirm the date table contains one row per date with no gaps.
2. Confirm the date column is used by the active relationship to the fact table.
3. Test a completed year, quarter, and month.
4. Test a partial year, quarter, and month.
5. Test a leap year and a month-end boundary.
6. Test a zero prior value and a blank prior value.
7. Test row-level results and the visual grand total.
8. For PP and POP, test year, quarter, month, and card contexts.
9. For complete-period patterns, apply a current-period slicer and verify that the prior complete period still resolves.
10. Record the expected and actual result in the pull request.
''', encoding='utf-8')
(root/'docs'/'definitions.md').write_text('''# Definitions

- YTD: Year-to-date
- QTD: Quarter-to-date
- MTD: Month-to-date
- MAT: Moving annual total
- PY: Previous year
- PQ: Previous quarter
- PM: Previous month
- PYC: Previous year complete
- PQC: Previous quarter complete
- PMC: Previous month complete
- PP: Previous period selected from hierarchy scope
- PYMAT: Previous-year moving annual total
- YOY: Year-over-year growth
- QOQ: Quarter-over-quarter growth
- MOM: Month-over-month growth
- MATG: Moving annual total growth
- POP: Period-over-period growth selected from hierarchy scope
- PYTD: Previous year-to-date
- PQTD: Previous quarter-to-date
- PMTD: Previous month-to-date
- YOYTD: Year-over-year-to-date growth
- QOQTD: Quarter-over-quarter-to-date growth
- MOMTD: Month-over-month-to-date growth
- YTDOPY: Year-to-date over complete previous year
- QTDOPQ: Quarter-to-date over complete previous quarter
- MTDOPM: Month-to-date over complete previous month
''', encoding='utf-8')

932

In [50]:
with (root/'manifest.csv').open('w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['code','name','description','kind','path'])
    w.writeheader(); w.writerows(patterns.values())
(root/'manifest.json').write_text(json.dumps(list(patterns.values()), indent=2), encoding='utf-8')

3991

In [ ]:
zip_path = Path(r"C:/Users/<USER>/<FOLDER>/dax-pattern-templates.zip")
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in root.rglob('*'):
        if p.is_file():
            z.write(p, Path(root.name)/p.relative_to(root))
print(f'Created {len(patterns)} patterns')
print(zip_path)